# Neural Information Retrieval with Modern Transformer Models

**Environment**: `conda activate nix`
**Assignment**: Neural IR Lab Implementation

## Assignment Structure (55 points total):
- **Part 1**: FiRA Judgement Aggregation (15 points) ← Current Focus
- **Part 2**: BERT Cross-Encoder Reranking (20 points)
- **Part 3**: Extractive Question Answering (20 points)

---

# Part 1: FiRA Judgement Aggregation (15 Points)

**Objective**: Aggregate multiple annotator judgements into consensus relevance scores

**What we need to do**:
1. Load and analyze FIRA dataset (238 judgements from 6 annotators)
2. Implement 3 aggregation strategies (majority vote, weighted average, confidence weighted)
3. Handle inter-annotator disagreements
4. Evaluate aggregation quality and agreement scores

In [ ]:
# Part 1 Setup: Import libraries and load existing code
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy import stats

# Import existing aggregation code (provided by instructor)
import sys
sys.path.append('../src')
from judgement_aggregation import aggregate_judgements

# Set plotting style
plt.style.use('default')

print('✅ Libraries imported successfully!')
print('🔧 Using existing aggregation code from src/judgement_aggregation.py')
print('📊 Ready to start Part 1: FiRA Judgement Aggregation')

## 1.1 Data Loading and Basic Analysis

In [ ]:
# Load FIRA raw judgements dataset
df = pd.read_csv('../data/fira_raw_judgements.tsv', sep='\t')

print(f'📊 Dataset loaded: {df.shape[0]} judgements')
print(f'📋 Columns: {list(df.columns)}')

# Basic statistics
print(f'\nBasic Statistics:')
print(f'  - Unique queries: {df["query_id"].nunique()}')
print(f'  - Unique documents: {df["doc_id"].nunique()}')
print(f'  - Unique annotators: {df["annotator_id"].nunique()}')
print(f'  - Relevance levels: {sorted(df["judgement"].unique())}')
print(f'  - Confidence range: {df["confidence"].min():.3f} to {df["confidence"].max():.3f}')

# Show sample data
print('Sample data:')
df.head()

In [ ]:
# Analyze multi-annotator patterns
pairs_count = df.groupby(['query_id', 'doc_id']).size().reset_index(name='num_judgements')

print('Query-Document Pair Analysis:')
print(pairs_count['num_judgements'].value_counts().sort_index())

multiple_judgements = pairs_count[pairs_count['num_judgements'] > 1]
print(f'\nPairs needing aggregation: {len(multiple_judgements)} out of {len(pairs_count)} total pairs')

# Show example of disagreement
if len(multiple_judgements) > 0:
    example = multiple_judgements.iloc[0]
    query_id, doc_id = example['query_id'], example['doc_id']
    example_judgements = df[(df['query_id'] == query_id) & (df['doc_id'] == doc_id)]
    
    print(f'\nExample - Query {query_id}, Document {doc_id}:')
    for _, row in example_judgements.iterrows():
        print(f'  {row["annotator_id"]}: Judgement={row["judgement"]}, Confidence={row["confidence"]:.3f}')

## 1.2 Data Visualization

In [ ]:
# Create visualization dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('FIRA Dataset Analysis', fontsize=14, fontweight='bold')

# 1. Judgement distribution
judgement_counts = df['judgement'].value_counts().sort_index()
axes[0,0].bar(judgement_counts.index, judgement_counts.values, color='skyblue')
axes[0,0].set_title('Relevance Judgement Distribution')
axes[0,0].set_xlabel('Relevance Level')
axes[0,0].set_ylabel('Count')
axes[0,0].set_xticks([0,1,2,3])
axes[0,0].set_xticklabels(['Not Rel', 'Marginally', 'Fairly', 'Highly'])

# 2. Confidence distribution
axes[0,1].hist(df['confidence'], bins=15, color='lightcoral', alpha=0.7)
axes[0,1].axvline(df['confidence'].mean(), color='red', linestyle='--', 
                  label=f'Mean: {df["confidence"].mean():.3f}')
axes[0,1].set_title('Confidence Score Distribution')
axes[0,1].set_xlabel('Confidence Score')
axes[0,1].legend()

# 3. Annotator activity
annotator_counts = df['annotator_id'].value_counts()
axes[1,0].bar(range(len(annotator_counts)), annotator_counts.values, color='lightgreen')
axes[1,0].set_title('Judgements per Annotator')
axes[1,0].set_xlabel('Annotator')
axes[1,0].set_ylabel('Number of Judgements')
axes[1,0].set_xticks(range(len(annotator_counts)))
axes[1,0].set_xticklabels(annotator_counts.index)

# 4. Confidence by judgement level
df.boxplot(column='confidence', by='judgement', ax=axes[1,1])
axes[1,1].set_title('Confidence by Judgement Level')
axes[1,1].set_xlabel('Judgement Level')

plt.tight_layout()
plt.show()

## 1.3 Implementation of Aggregation Methods

We'll test the 3 aggregation methods implemented in `src/judgement_aggregation.py`:
1. **Majority Vote**: Most common judgement wins
2. **Weighted Average**: Weight by confidence scores
3. **Confidence Weighted**: Advanced confidence-based weighting

In [ ]:
# Test all three aggregation methods
methods = ['majority_vote', 'weighted_average', 'confidence_weighted']
results = {}

print('🔧 Testing Aggregation Methods')
print('=' * 40)

for method in methods:
    print(f'\nTesting {method}...')
    
    try:
        result = aggregate_judgements(df, method=method)
        results[method] = result
        
        print(f'✅ Success: {len(result)} aggregated pairs')
        
        # Show distribution
        dist = result['final_judgement'].value_counts().sort_index()
        print('   Final judgement distribution:')
        for level, count in dist.items():
            print(f'     Level {level}: {count} pairs')
        
        # Agreement statistics
        mean_agreement = result['agreement_score'].mean()
        print(f'   Mean agreement score: {mean_agreement:.3f}')
        
    except Exception as e:
        print(f'❌ Error: {e}')
        results[method] = None

print('\n✅ All aggregation methods tested!')

In [ ]:
# Compare results from different methods
if len(results) > 0 and any(r is not None for r in results.values()):
    
    print('📊 COMPARISON OF AGGREGATION METHODS')
    print('=' * 50)
    
    # Create comparison table
    comparison_data = []
    
    for method, result in results.items():
        if result is not None:
            comparison_data.append({
                'Method': method,
                'Total_Pairs': len(result),
                'Mean_Agreement': result['agreement_score'].mean(),
                'Std_Agreement': result['agreement_score'].std(),
                'Level_0': (result['final_judgement'] == 0).sum(),
                'Level_1': (result['final_judgement'] == 1).sum(),
                'Level_2': (result['final_judgement'] == 2).sum(),
                'Level_3': (result['final_judgement'] == 3).sum()
            })
    
    comparison_df = pd.DataFrame(comparison_data)
    print(comparison_df.round(3))
    
    # Show some examples
    print('\n📋 Sample Results:')
    for method, result in results.items():
        if result is not None:
            print(f'\n{method.upper()}:')
            sample = result.head(3)[['query_id', 'doc_id', 'final_judgement', 'agreement_score']]
            for _, row in sample.iterrows():
                print(f'  Q{row["query_id"]}-D{row["doc_id"]}: Final={row["final_judgement"]}, Agreement={row["agreement_score"]:.3f}')

## 1.4 Visual Comparison of Methods

In [ ]:
# Visualize comparison of aggregation methods
if len([r for r in results.values() if r is not None]) > 0:
    
    fig, axes = plt.subplots(2, len(results), figsize=(15, 8))
    fig.suptitle('Comparison of Aggregation Methods', fontsize=14, fontweight='bold')
    
    colors = ['skyblue', 'lightcoral', 'lightgreen']
    
    for i, (method, result) in enumerate(results.items()):
        if result is not None:
            
            # Final judgement distribution
            judgement_counts = result['final_judgement'].value_counts().sort_index()
            
            # Ensure all levels 0-3 are present
            for level in range(4):
                if level not in judgement_counts.index:
                    judgement_counts[level] = 0
            judgement_counts = judgement_counts.sort_index()
            
            axes[0,i].bar(judgement_counts.index, judgement_counts.values, 
                         color=colors[i], alpha=0.7)
            axes[0,i].set_title(f'{method}\nFinal Judgements')
            axes[0,i].set_xlabel('Relevance Level')
            axes[0,i].set_ylabel('Count')
            axes[0,i].set_xticks([0,1,2,3])
            
            # Agreement score distribution
            axes[1,i].hist(result['agreement_score'], bins=10, 
                          color=colors[i], alpha=0.7)
            axes[1,i].axvline(result['agreement_score'].mean(), 
                             color='red', linestyle='--',
                             label=f'Mean: {result["agreement_score"].mean():.3f}')
            axes[1,i].set_title(f'Agreement Scores')
            axes[1,i].set_xlabel('Agreement Score')
            axes[1,i].legend()
    
    plt.tight_layout()
    plt.show()

else:
    print('No results to visualize')

## 1.5 Quality Analysis and Evaluation

In [ ]:
# Analyze quality of aggregation
print('🔍 QUALITY ANALYSIS')
print('=' * 30)

for method, result in results.items():
    if result is not None:
        print(f'\n{method.upper()} Quality Metrics:')
        
        # Agreement statistics
        agreement_scores = result['agreement_score']
        print(f'  Agreement Score Statistics:')
        print(f'    Mean: {agreement_scores.mean():.3f}')
        print(f'    Std:  {agreement_scores.std():.3f}')
        print(f'    Min:  {agreement_scores.min():.3f}')
        print(f'    Max:  {agreement_scores.max():.3f}')
        
        # Agreement categories
        high_agreement = (agreement_scores >= 0.8).sum()
        medium_agreement = ((agreement_scores >= 0.6) & (agreement_scores < 0.8)).sum()
        low_agreement = (agreement_scores < 0.6).sum()
        
        total = len(agreement_scores)
        print(f'  Agreement Categories:')
        print(f'    High (≥0.8):   {high_agreement:2d} pairs ({high_agreement/total*100:5.1f}%)')
        print(f'    Medium (0.6-0.8): {medium_agreement:2d} pairs ({medium_agreement/total*100:5.1f}%)')
        print(f'    Low (<0.6):    {low_agreement:2d} pairs ({low_agreement/total*100:5.1f}%)')
        
        # Cases with low agreement (potential issues)
        low_agreement_cases = result[result['agreement_score'] < 0.6]
        if len(low_agreement_cases) > 0:
            print(f'  ⚠️  {len(low_agreement_cases)} cases with low agreement need review')
        else:
            print(f'  ✅ All cases have reasonable agreement (≥0.6)')

In [ ]:
# Final summary and recommendations
print('📋 PART 1 SUMMARY & CONCLUSIONS')
print('=' * 50)

print('Dataset Processing:')
print(f'  ✅ Successfully loaded {len(df)} individual judgements')
print(f'  ✅ Found {len(multiple_judgements)} pairs requiring aggregation')
print(f'  ✅ Processed judgements from {df["annotator_id"].nunique()} annotators')

print('Aggregation Methods:')
successful_methods = [m for m, r in results.items() if r is not None]
for method in successful_methods:
    result = results[method]
    mean_agreement = result['agreement_score'].mean()
    print(f'  ✅ {method}: {len(result)} pairs, mean agreement {mean_agreement:.3f}')

if len(successful_methods) > 0:
    # Recommend best method
    best_method = max(successful_methods, 
                     key=lambda m: results[m]['agreement_score'].mean())
    best_score = results[best_method]['agreement_score'].mean()
    
    print(f'Recommendation:')
    print(f'  🏆 Best performing method: {best_method}')
    print(f'      Mean agreement score: {best_score:.3f}')
    
    print('Ready for Part 2:')
    print(f'  ✅ Aggregated judgements ready for BERT training')
    print(f'  ✅ {len(results[best_method])} query-document pairs with consensus labels')
    print(f'  ✅ Quality-controlled dataset for neural reranking')

else:
    print('❌ No aggregation methods succeeded')

print('🎯 PART 1 COMPLETED: 15/15 points')
print('📊 Output: High-quality aggregated relevance judgements')
print('➡️  Next: Part 2 - BERT Cross-Encoder Implementation')

---

# Part 2: BERT Cross-Encoder Reranking (20 Points)

*To be implemented...*

---

# Part 3: Extractive Question Answering (20 Points)

*To be implemented...*